# Patient-level attention VAE

8 latent variables, about 9.6 million parameters, spatial and temporal attention. Demographics join the latent mean **only at the MI classifier**. The decoder reconstructs sampled XYZ, using latent geometry alone.

Frame repair and sampling are patient-local. Scaling, model fitting and early stopping stay inside each fold; final test patients are excluded. See the README for assumptions and configuration.


In [ ]:
from pathlib import Path
from dataclasses import asdict
import json
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from mcpp.vae import CardiacVAE, VAEConfig
from mcpp.vae_data import DEMOGRAPHIC_COLUMNS, PreprocessConfig, load_cohort
from mcpp.vae_training import TrainConfig, cross_validate_vae, train_holdout_vae, mi_classification_report, select_device
from mcpp.vae_experiments import atomic_json_save

ROOT = Path.cwd() if (Path.cwd() / 'data').is_dir() else Path.cwd().parent
PREPROCESS = PreprocessConfig(n_points=384)
MODEL = VAEConfig(n_points=PREPROCESS.n_points, latent_dim=8)
N_FOLDS = 1  # 1 trains one model; 2+ runs cross-validation.
VALIDATION_FRACTION = .25 if N_FOLDS == 1 else .15
TRAINING = TrainConfig(epochs=1, batch_size=8, device="auto", amp=True, checkpoint_every=100,
                       early_stopping=False, inner_validation_fraction=VALIDATION_FRACTION)
TRAIN_SAMPLES = None  # Keep the original 60:20:20 split before quality exclusions.
LIMIT_PATIENTS = None  # e.g. 40 for a two-fold smoke run
EXPERIMENT = 'notebook-single-z8'  # Matching names continue; change for a fresh experiment.
OUTPUT = ROOT / 'results/vae_training' / EXPERIMENT
print('PyTorch:', torch.__version__, 'device:', select_device(TRAINING.device))
print('Parameters:', f'{CardiacVAE(MODEL).parameter_count():,}')
print('Head-only demographics:', DEMOGRAPHIC_COLUMNS)


## Prepare once

Retain 128 points in each anatomical region across all ten frames. Repair expansion and point scatter before sampling. Spatial coherence detects multi-frame defects; cycles with more than half their frames corrupted are excluded and recorded in the quality audit. This opens only the development patients' point arrays, one patient at a time. The existing 20% held-out test membership is preserved.


In [ ]:
metadata = pd.read_csv(ROOT / 'data/training_data/mixed_demographics.csv')
development, held_out = train_test_split(
    np.arange(len(metadata)), test_size=.2, stratify=metadata.MI, random_state=42,
)
original_train, original_validation = train_test_split(
    development, test_size=VALIDATION_FRACTION, stratify=metadata.MI.iloc[development], random_state=42,
)
if LIMIT_PATIENTS is not None and LIMIT_PATIENTS < len(development):
    development, _ = train_test_split(
        development, train_size=LIMIT_PATIENTS,
        stratify=metadata.MI.iloc[development], random_state=TRAINING.seed,
    )
xyz, regions, demographics, labels, repair_audit = load_cohort(
    ROOT / 'data/training_data', development, PREPROCESS,
)
development = np.asarray([a['patient_index'] for a in repair_audit if not a['excluded']], dtype=int)
quality_excluded = [a['patient_index'] for a in repair_audit if a['excluded']]
fixed_train_indices = np.flatnonzero(np.isin(development, original_train))
fixed_validation_indices = np.flatnonzero(np.isin(development, original_validation))
if N_FOLDS == 1 and TRAIN_SAMPLES is None:
    print('Retained 60:20:20 split: train', len(fixed_train_indices), 'validation', len(fixed_validation_indices), 'test', len(held_out))
flagged = pd.DataFrame([
    {'patient': a['patient_index'], 'frames': a['flagged_frames'], 'excluded': a['excluded']}
    for a in repair_audit if a['flagged_frames']
], columns=['patient', 'frames', 'excluded'])
print('XYZ:', xyz.shape, 'demographics:', demographics.shape)
print('Test patients excluded:', len(held_out), 'patients with suspect frames:', len(flagged), 'excluded cycles:', len(quality_excluded))
flagged


## Train one model or cross-validate

The default preserves the original 60:20:20 split, fixed before quality exclusions: 540/180/180 originally and 521/174/180 after excluding suspect development cycles. One model fits the training patients and reports validation MI F1 and accuracy. Set `N_FOLDS>=2` with a new experiment name for CV. Early stopping is disabled by default, so requested epochs are honoured; opt in explicitly if desired. Rolling resume checkpoints replace previous copies every 100 epochs and at invocation end. A separate `val_best.pt` (or `fold_N.val_best.pt` in CV) is replaced immediately on every strict improvement in validation macro F1. `training.log` appends training output across runs. Final reports still use the lowest-validation-BCE model. Use a new experiment name when changing latent dimensions.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
single_fold = N_FOLDS == 1
runner = train_holdout_vae if single_fold else cross_validate_vae
split_options = {'train_samples': TRAIN_SAMPLES, 'validation_fraction': VALIDATION_FRACTION} if single_fold else {'n_splits': N_FOLDS}
if single_fold and TRAIN_SAMPLES is None:
    split_options.update(train_indices=fixed_train_indices, validation_indices=fixed_validation_indices)
results = runner(
    xyz, regions, demographics, labels, **split_options,
    model_config=MODEL, train_config=TRAINING, preprocess_config=PREPROCESS,
    output_dir=OUTPUT, patient_ids=development, experiment_name=EXPERIMENT,
)
atomic_json_save(repair_audit, OUTPUT / 'frame_quality.json')
scored = np.asarray(results['scored_indices'], dtype=int)
score_labels, score_patients = labels[scored], development[scored]
prefix = 'validation' if single_fold else 'oof'
probabilities = results[f'{prefix}_probabilities']
np.savez_compressed(
    OUTPUT / f'{prefix}_predictions.npz', patient_ids=score_patients, labels=score_labels,
    probabilities=probabilities, latent_means=results[f'{prefix}_latent_means'],
)
summary = {key: value for key, value in results.items()
           if key not in (f'{prefix}_probabilities', f'{prefix}_latent_means')}
summary.update(model_config=asdict(MODEL), train_config=asdict(TRAINING),
               preprocess_config=asdict(PREPROCESS), test_set_scored=False,
               test_patient_ids_excluded=held_out.tolist(),
               quality_excluded_patient_ids=quality_excluded,
               demographic_columns=list(DEMOGRAPHIC_COLUMNS),
               classification_threshold=.5, latent_axes_aligned_across_folds=False)
if single_fold and TRAIN_SAMPLES is None:
    summary.update(original_split_counts={'train': len(original_train), 'validation': len(original_validation), 'test': len(held_out)},
                   original_train_patient_ids=original_train.tolist(), original_validation_patient_ids=original_validation.tolist())
atomic_json_save(summary, OUTPUT / ('holdout_results.json' if single_fold else 'cv_results.json'))
mi_report = mi_classification_report(score_labels, probabilities)
mi_report.update(evaluation=results['evaluation'],
                 n_folds=N_FOLDS, epochs_per_fold=max(f['completed_epochs'] for f in results['folds']),
                 epochs_added_per_invocation=TRAINING.epochs, experiment_name=EXPERIMENT,
                 completed_epochs_per_fold=[f['completed_epochs'] for f in results['folds']],
                 training_patients_per_fold=[len(f['fit_patient_ids']) for f in results['folds']],
                 validation_patients_per_fold=[len(f['early_stop_patient_ids']) for f in results['folds']],
                 validation_used_for_model_selection=single_fold,
                 early_stopping_enabled=TRAINING.early_stopping,
                 stop_reasons=[f['stop_reason'] for f in results['folds']], test_set_scored=False,
                 n_quality_excluded=len(quality_excluded), n_test_held_out=len(held_out),
                 quality_excluded_patient_ids=quality_excluded)
atomic_json_save(mi_report, OUTPUT / 'mi_classification_report.json')
patient_folds = {patient: fold['fold'] for fold in results['folds'] for patient in fold['score_patient_ids']}
pd.DataFrame({
    'patient_index': score_patients,
    'fold': [patient_folds[int(patient)] for patient in score_patients],
    'true_MI': np.where(score_labels, 'pMI', 'healthy'),
    'predicted_MI': np.where(probabilities >= .5, 'pMI', 'healthy'),
    'probability_pMI': probabilities,
}).to_csv(OUTPUT / 'mi_predictions.csv', index=False)
print(f"MI ({results['evaluation']}): F1(pMI)={mi_report['metrics']['f1_pMI']:.4f}; "
      f"accuracy={mi_report['metrics']['accuracy']:.2%}; scored patients={mi_report['n_patients']}")
print('Saved report:', OUTPUT / 'mi_classification_report.json')
pd.DataFrame([{'fold': f['fold'], 'best_epoch': f['best_epoch'], **f['metrics']}
              for f in results['folds']])


In [ ]:
pd.DataFrame({
    'mean across models': results['mean_fold_metrics'],
    'model standard deviation': results['std_fold_metrics'],
    results['evaluation']: results['validation_metrics'] if single_fold else results['oof_metrics'],
})


Each epoch prints training and inner-validation accuracy, pMI F1 and macro F1 at cutoff 0.5, alongside the losses and elapsed time. The scores are saved in each fold's `history` as `train_accuracy`, `train_f1_pMI`, `train_macro_f1`, `validation_accuracy`, `validation_f1_pMI` and `validation_macro_f1`. Training scores pool predictions from the training pass; validation scores use the epoch-end model with dropout disabled. Check reconstruction, KL and classification losses for collapse or overfitting. The saved checkpoints contain the demographic order, class mapping, geometry preprocessing, and train-fitted scaling. `predict_patients` in `mcpp.vae_training` accepts raw patient cycles and head-only demographics. Independent folds learn different latent coordinate systems; their latent axes are not aligned.
